<a href="https://colab.research.google.com/github/Boulder1-kihara/study-helper/blob/main/study_helper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install langchain langchain-community langchain-chroma sentence-transformers
!pip install -qU langchain langchain-community langchain-core langchain-text-splitters chromadb sentence-transformers duckduckgo-search langchain-google-genai pypdf


In [ ]:
!pip install unstructured

from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
import os

# Create a dummy folder and a dummy PDF for demonstration
if not os.path.exists('./my_notes_folder'):
    os.makedirs('./my_notes_folder')

# Create a dummy PDF file
with open('./my_notes_folder/dummy_note.pdf', 'w') as f:
    f.write('This is a dummy PDF file content.')
    f.write('\nThis is the second line of the dummy PDF.')
    f.write('\nMachine learning is a field of artificial intelligence.')
    f.write('\nNatural Language Processing (NLP) is a subfield of AI.')
    f.write('\nVector databases are important for RAG applications.')

# 1. Load your notes
loader = DirectoryLoader('./my_notes_folder', glob="**/*.pdf")
documents = loader.load()

# 2. Split them into manageable chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

# 3. Create local embeddings and store them in ChromaDB
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db")
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print(f"Loaded {len(documents)} document(s).")
print(f"Split into {len(chunks)} chunks.")

In [ ]:
!pip install -U ddgs

from langchain_core.tools import create_retriever_tool
from langchain_community.tools import DuckDuckGoSearchRun

# Tool 1: The Local Notes Search
notes_tool = create_retriever_tool(
    retriever,
    name="university_notes_search",
    description="Search for information, definitions, and concepts from the user's official class notes. Always use this tool first."
)

# Tool 2: The Web Fallback
web_search = DuckDuckGoSearchRun()
web_tool = web_search

tools = [notes_tool, web_tool]

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

# 1. Initialize the LLM with the CURRENT active model version
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", max_retries=5, google_api_key="api key")

# 2. Define your system instructions
system_prompt = (
    "you were created by Abel. this is your enginer. your name is 'university study helper'"
    "You are a helpful university tutor. Answer the student's past paper questions. "
    "First, search the 'university_notes_search' tool. If the answer is complete, use it. "
    "If the notes do not contain the answer or are missing details, use the web search tool to find the rest."
)

# 3. Create the LangGraph Agent
agent_executor = create_react_agent(llm, tools, prompt=system_prompt)

In [ ]:
from IPython.display import display, Markdown

# The question from your past paper
question = input("Type your question here: ")

print("Searching for the answer...\n")

# Invoke the agent
response = agent_executor.invoke({"messages": [("human", question)]})

# Extract the raw content from the final message
raw_content = response["messages"][-1].content

# Check if the response is a list of blocks and extract the text
if isinstance(raw_content, list):
    final_text = raw_content[0].get("text", "")
else:
    # Fallback just in case it returns a normal string
    final_text = raw_content

# Display the output with clean formatting
display(Markdown(final_text))